# n000 — General Statistics

This notebook computes and visualises high-level statistics across all experiment conditions.

**Outputs** (saved to `<ROOT>/plots/`):
- `000_general_stats_boxplots.pdf` — per-metric boxplots
- `000_general_stats.pdf` — combined 4-panel figure (main paper figure)
- `000_general_pareto.pdf` — 2D Pareto scatter plots
- `000_general_pareto_3d_means.pdf` — 3D Pareto surface over experiment means
- `000_agents_food.pdf` — agent count and food count time series
- `000_deaths.pdf` — cumulative death-cause breakdown over time
- `000_actions.pdf` — per-action cumulative rates over time

**Metrics tracked per experiment run:**
- Experiment length (population longevity)
- Total agents born
- Total artifacts created/expired
- Population size over time
- Food availability over time
- Deaths by old age vs. hunger
- Per-action counts (give, take, reproduce, move, artifact interactions, …)

In [ ]:
import numpy as np
import json
import glob
import matplotlib.pyplot as plt
from core.utils.analysis_utils import get_exp_folders, get_last_ts, agents_time, load_worldlog, load_agent_log
from core.utils import ROOT
import matplotlib.pyplot as plt
from tqdm import tqdm
# %matplotlib widget

# force reload these everytime
import importlib
importlib.reload(importlib.import_module('analysis_scripts.plot_utils'))
from analysis_scripts.plot_utils import color_map, plot_params

plt.rcParams.update(plot_params)

## Configuration

Set `EXP_BASE_NAMES` to the list of experiment condition names you want to compare. Each name must correspond to a subdirectory pattern under `<ROOT>/logs/`.

- `PLOT_NAMES` maps internal names to display labels used in plot titles and legends.
- `SHOW_STD` toggles between shaded ±1 std bands (`True`) and individual run traces (`False`) on time-series plots.
- Only **completed** runs are loaded (via `only_completed=True`).

In [ ]:
EXP_BASE_NAMES = [] # TO DEFINE

# Change this if you want other names in the plot
PLOT_NAMES = {exp_name: exp_name.replace("_", " ").title() for exp_name in EXP_BASE_NAMES}

EXP_PATH = ROOT / "logs"
SHOW_STD = False

experiments = {}
for base_name in EXP_BASE_NAMES:
    experiments[base_name] = get_exp_folders(log_path=EXP_PATH, 
                                             exp_name=base_name, 
                                             only_completed=True)

## Data Loading

For each experiment condition and each completed run, load:
- **Experiment length** — last timestamp in `open_gridworld.log`
- **Total agents** — count from `agent_names.json`
- **Total artifacts** — sum of active + expired from `artifacts.json`
- **Agents per timestep** — population time series from the world log
- **Food per timestep** — food count time series from `food_counts.json`

Results are stored in `data[base_name]` as lists (one entry per run).

In [ ]:
data = {}
for base_name, exps in experiments.items():
    exp_lengths = []
    total_agents = []
    total_artifacts = []
    agents_per_time = []
    food_per_time = []
    for exp in tqdm(exps, desc=base_name):
        worldlog_path = EXP_PATH / exp / 'open_gridworld.log'
        exp_lengths.append(get_last_ts(worldlog_path=worldlog_path))

        names_path = EXP_PATH / exp / "agent_names.json"
        if names_path.exists():
            total_agents.append(len(json.load(open(names_path))))

        artifacts_path = EXP_PATH / exp / "artifacts.json"
        if artifacts_path.exists():
            artifacts = json.load(open(artifacts_path))
            total_artifacts.append(len(artifacts['active']) + len(artifacts['expired']))

        agents_per_time.append(agents_time(worldlog_path=worldlog_path))
        food_path = EXP_PATH / exp / "food_counts.json"
        if food_path.exists():
            food = np.array(json.load(open(food_path)))
            food_per_time.append(food)


    data[base_name] = {'exp_lengths': exp_lengths, 'total_agents':total_agents, 'total_artifacts': total_artifacts,
                       'agents_per_time': agents_per_time, 'food_per_time': food_per_time
                       }

## Boxplots — Summary Statistics

Defines `make_boxplot()`, a reusable helper that draws a styled boxplot for one metric across all conditions. Features:
- IQR whiskers (`whis=1`), no outlier fliers
- Translucent coloured boxes; median line and mean marker at full opacity
- Colours sourced from the shared `color_map`

Then builds a tidy long-format DataFrame (`boxplot_df`) with columns `experiment`, `metric`, and `value`, computing:
- **Experiment Length** — raw run duration
- **Average Artifacts per Agent** — `total_artifacts / total_agents`
- **Average Population Size** — `Σ agents_per_time / exp_length`

Produces `000_general_stats_boxplots.pdf` (3 panels).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import itertools
import os

def make_boxplot(
    ax,
    df,
    metric,
    labels,
    palette,
    y_label,
    x_labels,
    whis=1,
    box_alpha=0.5,
    title=None,
):
    positions = []
    data_groups = []
    colors = []

    # build groups
    for i, exp in enumerate(labels):
        vals = df[(df["metric"] == metric) & (df["experiment"] == exp)]["value"].dropna().values
        positions.append(i)
        data_groups.append(vals)
        colors.append(palette[i])

    # draw the grouped boxplot
    bp = ax.boxplot(
        data_groups,
        positions=positions,
        patch_artist=True,
        showfliers=False,
        showcaps=False,
        whis=whis,
        medianprops={"linewidth": 2},
        whiskerprops={"linewidth": 1},
        capprops={"linewidth": 1},
        widths=0.8,
    )

    # mean markers
    for x, vals in zip(positions, data_groups):
        mean_val = float(np.mean(vals)) if len(vals) else np.nan
        ax.plot(
            x, mean_val,
            marker="o",
            markersize=6,
            markerfacecolor="white",
            markeredgecolor="black",
            markeredgewidth=1.3,
            zorder=5,
        )

    # style: translucent face, no edges
    for patch, c in zip(bp["boxes"], colors):
        r, g, b = mcolors.to_rgb(c)
        patch.set_facecolor((r, g, b, box_alpha))
        patch.set_edgecolor("none")

    # medians full opacity
    for med, c in zip(bp["medians"], colors):
        r, g, b = mcolors.to_rgb(c)
        med.set_color((r, g, b, 1.0))
        med.set_linewidth(2)

    # whiskers/caps matched per box — two per experiment
    rep_colors = list(itertools.chain.from_iterable([[c, c] for c in colors]))

    for whisk, c in zip(bp["whiskers"], rep_colors):
        r, g, b = mcolors.to_rgb(c)
        whisk.set_color((r, g, b, 1.0))

    # (cap styling included for completeness, even though showcaps=False)
    for cap, c in zip(bp["caps"], rep_colors):
        r, g, b = mcolors.to_rgb(c)
        cap.set_color((r, g, b, 1.0))

    # axes formatting
    if title is not None:
        ax.set_title(title)
    else:
        ax.set_title(metric)
    ax.set_ylabel(y_label)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(x_labels, rotation=45, ha="right")
    ax.grid(alpha=0.3)

    return bp


# -----------------------------
# Original code, refactored
# -----------------------------
metrics = ["Experiment Length", "Average Population Size", "Average Artifacts per Agent"]
y_labels = ["Time Steps", "Number of Agents", "Artifacts Count"]
labels = list(data.keys())
n_experiments = len(labels)

rows = []
for exp in labels:
    for exp_len, agents, artifacts, agents_per_time in zip(
        data[exp]["exp_lengths"],
        data[exp]["total_agents"],
        data[exp]["total_artifacts"],
        data[exp]["agents_per_time"],
    ):
        exp_len = float(exp_len)
        agents = float(agents)
        artifacts = float(artifacts)

        rows.append({"experiment": exp, "metric": "Experiment Length", "value": exp_len})

        avg_art_per_agent = artifacts / agents if agents != 0 else 0.0
        rows.append({"experiment": exp, "metric": "Average Artifacts per Agent", "value": avg_art_per_agent})

        avg_pop_size = float(np.sum(agents_per_time)) / exp_len if exp_len != 0 else 0.0
        rows.append({"experiment": exp, "metric": "Average Population Size", "value": avg_pop_size})

        rows.append({"experiment": exp, "metric": "Total Artifacts", "value": artifacts})

boxplot_df = pd.DataFrame(rows)

# palette extracted from your color_map, in correct experiment order
palette = [color_map[bn] for bn in labels]

# x tick labels (as in your code)
x_labels = [PLOT_NAMES[bn] for bn in EXP_BASE_NAMES]

# boxplot layout parameters
fig, axes = plt.subplots(1, 3, figsize=(10, 4), sharey=False)

for ax, metric, ylab in zip(axes, metrics, y_labels):
    make_boxplot(
        ax=ax,
        df=boxplot_df,
        metric=metric,
        labels=labels,
        palette=palette,
        y_label=ylab,
        x_labels=x_labels,
        whis=1,
        box_alpha=0.5,
    )

plt.tight_layout()
os.makedirs(ROOT / "plots", exist_ok=True)
plt.savefig(ROOT / "plots" / "000_general_stats_boxplots.pdf", dpi=300)
plt.show()

## 2D Pareto Scatter Plots

Defines two helpers:

- **`pareto_mask_2d(points, maximize)`** — returns a boolean mask of Pareto-efficient points for any mix of maximisation/minimisation objectives.
- **`plot_pareto(ax, runs_df, xcol, ycol, ...)`** — scatter-plots per-run points (faint), per-experiment means with error bars (IQR by default), and a Pareto frontier line through the experiment means.

Error-bar style is controlled by the `whiskers` argument (`"std"`, `"sem"`, `"ci95"`, or `"iqr"`).

Produces `000_general_pareto.pdf` with three 2D views:
1. Avg population vs avg artifacts/agent
2. Experiment length vs avg population
3. Experiment length vs avg artifacts/agent

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# -----------------------------
# 1) Build ONE row per run
# -----------------------------
records = []
for exp in labels:
    for run_i, (exp_len, agents, artifacts, agents_per_time) in enumerate(
        zip(
            data[exp]["exp_lengths"],
            data[exp]["total_agents"],
            data[exp]["total_artifacts"],
            data[exp]["agents_per_time"],
        )
    ):
        exp_len = float(exp_len)
        agents = float(agents)
        artifacts = float(artifacts)

        avg_art_per_agent = artifacts / agents if agents != 0 else 0.0
        avg_pop_size = float(np.sum(agents_per_time)) / exp_len if exp_len != 0 else 0.0

        records.append(
            {
                "experiment": exp,
                "run": run_i,
                "exp_len": exp_len,
                "avg_pop": avg_pop_size,
                "avg_art_per_agent": avg_art_per_agent,
                "total_artifacts": artifacts,
            }
        )

pareto_df = pd.DataFrame(records)

# -----------------------------
# 2) Pareto helper (2D)
#    maximize flags per dim
# -----------------------------
def pareto_mask_2d(points: np.ndarray, maximize=(True, True)) -> np.ndarray:
    """
    points: (N,2) array
    maximize: tuple(bool,bool) where True means maximize that dimension, False means minimize
    returns: boolean mask of Pareto-efficient points
    """
    pts = points.astype(float).copy()

    # Convert to a "maximize everything" problem by flipping minimization dims
    for d, mx in enumerate(maximize):
        if not mx:
            pts[:, d] *= -1.0

    n = pts.shape[0]
    is_eff = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_eff[i]:
            continue
        # A point is dominated if there exists another point >= in all dims and > in at least one.
        dominates_i = (pts >= pts[i]).all(axis=1) & (pts > pts[i]).any(axis=1)
        dominates_i[i] = False
        if dominates_i.any():
            is_eff[i] = False
    return is_eff

def plot_pareto(
    ax, runs_df, xcol, ycol, xlab, ylab,
    maximize=(True, True),
    show_runs=True,
    whiskers="iqr",          # "std", "sem", or "ci95"
    whisker_alpha=0.5,
    whisker_lw=1.,
    whisker_capsize=3,
):
    import numpy as np

    # 1) Optionally show per-run cloud (faint)
    if show_runs:
        for exp, c in zip(labels, palette):
            sub = runs_df[runs_df["experiment"] == exp]
            ax.scatter(
                sub[xcol], sub[ycol],
                s=25,
                alpha=0.25,
                facecolors=mcolors.to_rgba(c, 0.35),
                edgecolors="none",
                zorder=1,
            )

    # 2) Aggregate: mean per experiment
    mean_df = (
        runs_df
        .groupby("experiment", as_index=False)[["exp_len", "avg_pop", "avg_art_per_agent", "total_artifacts"]]
        .mean()
    )

    # 2b) Std / SEM / CI per experiment for xcol and ycol
    stats = (
        runs_df
        .groupby("experiment")[[xcol, ycol]]
        .agg(["mean", "std", "count"])
    )

    q_df = (
        runs_df
        .groupby("experiment")[[xcol, ycol]]
        .quantile([0.25, 0.75])
        .unstack(level=1)  # columns like (xcol, 0.25), (xcol, 0.75), ...
    )

    # 3) Plot means + whiskers
    for exp, c in zip(labels, palette):
        if exp not in stats.index:
            continue

        mx = float(stats.loc[exp, (xcol, "mean")])
        my = float(stats.loc[exp, (ycol, "mean")])

        sx = float(stats.loc[exp, (xcol, "std")]) if not np.isnan(stats.loc[exp, (xcol, "std")]) else 0.0
        sy = float(stats.loc[exp, (ycol, "std")]) if not np.isnan(stats.loc[exp, (ycol, "std")]) else 0.0

        n = int(stats.loc[exp, (xcol, "count")])

        if whiskers == "std":
            ex, ey = sx, sy

        elif whiskers == "sem":
            ex = sx / np.sqrt(n) if n > 0 else 0.0
            ey = sy / np.sqrt(n) if n > 0 else 0.0

        elif whiskers == "ci95":
            # 95% CI of the mean: t * SEM (small-n friendly)
            # For n=5, t≈2.776
            t = 2.776 if n == 5 else 2.0
            ex = t * (sx / np.sqrt(n)) if n > 1 else 0.0
            ey = t * (sy / np.sqrt(n)) if n > 1 else 0.0

        elif whiskers == "iqr" and exp in q_df.index:
            x_q1 = float(q_df.loc[exp, (xcol, 0.25)])
            x_q3 = float(q_df.loc[exp, (xcol, 0.75)])
            y_q1 = float(q_df.loc[exp, (ycol, 0.25)])
            y_q3 = float(q_df.loc[exp, (ycol, 0.75)])


            left  = max(0.0, mx - x_q1)
            right = max(0.0, x_q3 - mx)
            down  = max(0.0, my - y_q1)
            up    = max(0.0, y_q3 - my)
            ex = [[left], [right]]
            ey = [[down], [up]]

        else:
            raise ValueError('whiskers must be one of: "std", "sem", "ci95"')

        # errorbar whiskers
        ax.errorbar(
            mx, my,
            xerr=ex,
            yerr=ey,
            fmt="none",
            ecolor=mcolors.to_rgba(c, whisker_alpha),
            elinewidth=whisker_lw,
            capsize=whisker_capsize,
            capthick=whisker_lw,
            zorder=2,
        )

        # mean marker
        ax.scatter(
            mx, my,
            s=110,
            alpha=1.0,
            facecolors=mcolors.to_rgba(c, 1.0),
            edgecolors="black",
            linewidths=1.2,
            zorder=3,
            label=PLOT_NAMES.get(exp, exp),
        )

    # 4) Pareto on experiment means
    pts = mean_df[[xcol, ycol]].to_numpy()
    mask = pareto_mask_2d(pts, maximize=maximize)
    pareto_mean = mean_df.loc[mask].copy()

    # 5) Pareto line
    pareto_pts = pareto_mean[[xcol, ycol]].to_numpy()
    order = np.argsort(pareto_pts[:, 0])
    ax.plot(
        pareto_pts[order, 0],
        pareto_pts[order, 1],
        color="black",
        linewidth=1.8,
        zorder=4,
        label="Pareto",
    )

    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.grid(alpha=0.3)
    return mean_df, pareto_mean
# -----------------------------
# 3) Make the 3 Pareto subplots
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)

# (A) avg population vs avg artifacts/agent  (maximize, maximize)
plot_pareto(
    axes[0],
    pareto_df,
    xcol="avg_pop",
    ycol="avg_art_per_agent",
    xlab="Average Population Size",
    ylab="Average Artifacts per Agent",
    maximize=(True, True),
)
axes[0].set_title("Pareto: Avg Pop vs Avg Artifacts/Agent")

# (B) experiment length vs avg population  (minimize length, maximize pop)
plot_pareto(
    axes[1],
    pareto_df,
    xcol="exp_len",
    ycol="avg_pop",
    xlab="Experiment Length (Time Steps)",
    ylab="Average Population Size",
    maximize=(True, True),
)
axes[1].set_title("Pareto: Length vs Avg Pop")

# (C) experiment length vs avg artifacts/agent  (minimize length, maximize artifacts/agent)
plot_pareto(
    axes[2],
    pareto_df,
    xcol="exp_len",
    ycol="avg_art_per_agent",
    xlab="Experiment Length (Time Steps)",
    ylab="Average Artifacts per Agent",
    maximize=(True, True),
)
axes[2].set_title("Pareto: Length vs Avg Artifacts/Agent")

# one legend for the whole figure (optional)
# the legend should go lower, below the plots, as of now it overlaps with the plots
handles, leg_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, leg_labels, loc="lower center", ncol=4, frameon=True,
           bbox_to_anchor=(0.5, -0.1)  # push legend below the figure
)

# leave space at bottom for legend
plt.tight_layout(rect=[0, 0.08, 1, 1])

# plt.tight_layout()
os.makedirs(ROOT / "plots", exist_ok=True)
plt.savefig(ROOT / "plots" / "000_general_pareto.pdf", dpi=300)
plt.show()

## Combined Figure (Main Paper Figure)

4-panel figure combining the most informative views into a single publication-ready layout:
- **(a)** Population Longevity — boxplot of experiment length
- **(b)** Artifact Count — boxplot of total artifacts
- **(c)** Population size vs Artifacts/Agent — Pareto scatter (`maximize=(False, True)`: minimise pop size, maximise cultural output)
- **(d)** Population Longevity vs Artifacts/Agent — Pareto scatter

Legend is placed at bottom-right spanning the Pareto panels. Saved as `000_general_stats.pdf`.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)

# (A) Experiment Length boxplot
make_boxplot(
        ax=axes[0],
        df=boxplot_df,
        metric="Experiment Length",
        labels=labels,
        palette=palette,
        y_label='Population Longevity (Time Steps)',
        x_labels=x_labels,
        whis=1,
        box_alpha=0.5,
        title="(A) Experiment Length",
    )
axes[0].set_title(r"$\bf{a}$ Population Longevity", loc='left')

make_boxplot(
        ax=axes[1],
        df=boxplot_df,
        metric="Total Artifacts",
        labels=labels,
        palette=palette,
        y_label='Artifact Count',
        x_labels=x_labels,
        whis=1,
        box_alpha=0.5,
        title="(B) Total Artifacts",
    )
axes[1].set_title(r"$\bf{b}$ Artifact Count", loc='left')

# (B) Pareto plot: avg population vs avg artifacts/agent  (maximize, maximize)
plot_pareto(
    axes[2],
    pareto_df,
    xcol="avg_pop",
    ycol="avg_art_per_agent",
    xlab="Average Population Size",
    ylab="Average Artifacts per Agent",
    maximize=(False, True),
)
axes[2].set_title(r"$\bf{c}$ Pop. size vs Artifacts/Agent", loc='left')


# (C) experiment length vs avg artifacts/agent  (minimize length, maximize artifacts/agent)
plot_pareto(
    axes[3],
    pareto_df,
    xcol="exp_len",
    ycol="avg_art_per_agent",
    xlab="Population Longevity(Time Steps)",
    ylab="Average Artifacts per Agent",
    maximize=(True, True),
)
axes[3].set_title(r"$\bf{d}$ Pop. Longevity vs Artifacts/Agent", loc='left')

handles, leg_labels = axes[2].get_legend_handles_labels()
fig.legend(handles, leg_labels, loc="lower center", ncol=3, frameon=False,
           bbox_to_anchor=(0.75, -0.02)  # push legend below the figure
)

# leave space at bottom for legend
# plt.tight_layout(rect=[0, 0.08, 1, 1])

plt.tight_layout()
os.makedirs(ROOT / "plots", exist_ok=True)
plt.savefig(ROOT / "plots" / "000_general_stats.pdf", dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)

plot_pareto(
    axes[0],
    pareto_df,
    xcol="exp_len",
    ycol="total_artifacts",
    xlab="Population Longevity (Time Steps)",
    ylab="Total Artifacts",
    maximize=(True, True),
)
axes[0].set_title(r"$\bf{a}$ Pop. Longevity vs Total Artifacts", loc='left')

# (B) Pareto plot: avg population vs avg artifacts/agent  (maximize, maximize)
plot_pareto(
    axes[1],
    pareto_df,
    xcol="avg_pop",
    ycol="avg_art_per_agent",
    xlab="Average Population Size",
    ylab="Average Artifacts per Agent",
    maximize=(False, True),
)
axes[1].set_title(r"$\bf{b}$ Pop. size vs Artifacts/Agent", loc='left')


# (C) experiment length vs avg artifacts/agent  (minimize length, maximize artifacts/agent)
plot_pareto(
    axes[2],
    pareto_df,
    xcol="exp_len",
    ycol="avg_art_per_agent",
    xlab="Population Longevity (Time Steps)",
    ylab="Average Artifacts per Agent",
    maximize=(True, True),
)
axes[2].set_title(r"$\bf{c}$ Pop. Longevity vs Artifacts/Agent", loc='left')

handles, leg_labels = axes[2].get_legend_handles_labels()
fig.subplots_adjust(bottom=0.22)

fig.legend(handles, leg_labels, loc="lower center", ncol=5, frameon=False,
           bbox_to_anchor=(0.53, -0.15)  # push legend below the figure
)

# leave space at bottom for legend
# plt.tight_layout(rect=[0, 0.08, 1, 1])

plt.tight_layout()
os.makedirs(ROOT / "plots", exist_ok=True)
plt.savefig(ROOT / "plots" / "000_general_stats.pdf", dpi=300, bbox_inches="tight")
plt.show()

## 3D Pareto Visualisation

Extends Pareto analysis to three simultaneous objectives using **`pareto_mask_nd`** (N-dimensional generalisation).

Objectives: maximise experiment length, maximise average population, maximise average artifacts/agent.

- All experiment means are plotted as coloured spheres.
- Pareto-efficient means are highlighted with larger markers and thicker edges.
- A semi-transparent triangulated surface is fitted through the Pareto front (skipped if fewer than 3 non-collinear points).
- Drop-lines connect each Pareto point to the coordinate planes for depth perception.

**`add_droplines_3d(ax, x, y, z)`** is a helper that draws three projection lines to the XY, XZ, and YZ planes.

Saved as `000_general_pareto_3d_means.pdf`.

In [ ]:
def pareto_mask_nd(points: np.ndarray, maximize: tuple) -> np.ndarray:
    """
    points: (N,D)
    maximize: tuple of length D; True => maximize that dim, False => minimize
    """
    pts = points.astype(float).copy()

    # convert to "maximize all" by flipping minimization dims
    for d, mx in enumerate(maximize):
        if not mx:
            pts[:, d] *= -1.0

    n = pts.shape[0]
    is_eff = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_eff[i]:
            continue
        # point i is dominated if ∃ j: pts[j] >= pts[i] in all dims and > in at least one
        dominated_by_any = (pts >= pts[i]).all(axis=1) & (pts > pts[i]).any(axis=1)
        dominated_by_any[i] = False
        if dominated_by_any.any():
            is_eff[i] = False
    return is_eff

def add_droplines_3d(ax, x, y, z, x0=None, y0=None, z0=None, alpha=1, lw=1.0):
    """
    Draws 3 lines from point (x,y,z) to the coordinate planes:
      - down to XY plane (z=z0)
      - over to XZ plane (y=y0)
      - over to YZ plane (x=x0)
    """

    # default to axis minima (nice visual anchor)
    if x0 is None:
        x0 = ax.get_xlim3d()[0]
    if y0 is None:
        y0 = ax.get_ylim3d()[1]
    if z0 is None:
        z0 = ax.get_zlim3d()[0]

    # to XY plane
    ax.plot([x, x], [y, y], [z0, z], alpha=alpha, linewidth=lw, color="black")

    # to XZ plane
    ax.plot([x, x], [y0, y], [z, z], alpha=alpha, linewidth=lw, color="black")

    # to YZ plane
    ax.plot([x0, x], [y, y], [z, z], alpha=alpha, linewidth=lw, color="black")

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

mean_df = (
    pareto_df
    .groupby("experiment", as_index=False)[["exp_len", "avg_pop", "avg_art_per_agent"]]
    .mean()
)

# Objectives:
# - minimize exp_len
# - maximize avg_pop
# - maximize avg_art_per_agent
pts3 = mean_df[["exp_len", "avg_pop", "avg_art_per_agent"]].to_numpy()
mask3 = pareto_mask_nd(pts3, maximize=(True, False, True))
pareto3_df = mean_df.loc[mask3]

print("Total points:", len(mean_df))
print("Pareto points:", len(pareto3_df))

# Check duplicates in the base plane you triangulate on (here X=exp_len, Y=avg_art_per_agent)
xy = pareto3_df[["exp_len", "avg_art_per_agent"]].to_numpy()
print("Pareto unique (exp_len, avg_art_per_agent):", len(np.unique(xy, axis=0)))

# Check collinearity in that plane (rank < 2 => collinear)
X = pareto3_df["exp_len"].to_numpy()
Y = pareto3_df["avg_pop"].to_numpy()
rank_xy = np.linalg.matrix_rank(np.column_stack([X - X.mean(), Y - Y.mean()])) if len(X) >= 2 else 0
print("Rank in (exp_len, avg_pop) plane:", rank_xy)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")

# All experiment means (colored)
for exp, c in zip(labels, palette):
    sub = mean_df[mean_df["experiment"] == exp]
    ax.scatter(
        sub["exp_len"], sub["avg_pop"], sub["avg_art_per_agent"],
        s=80,
        alpha=0.9,
        facecolors=mcolors.to_rgba(c, 0.9),
        edgecolors="black",
        linewidths=0.8,
        label=PLOT_NAMES.get(exp, exp),
    )

from matplotlib.tri import Triangulation

# pareto_df already computed, with columns: exp_len, avg_pop, avg_art_per_agent

X = pareto3_df["exp_len"].to_numpy()
Y = pareto3_df["avg_pop"].to_numpy()
Z = pareto3_df["avg_art_per_agent"].to_numpy()

# Remove duplicates in (X,Y) (Triangulation can choke on exact duplicates)
xy = np.column_stack([X, Y])
_, uniq_idx = np.unique(xy, axis=0, return_index=True)
X, Y, Z = X[uniq_idx], Y[uniq_idx], Z[uniq_idx]

# Need at least 3 points to plot triangles
if len(X) >= 3:
    # Check non-collinearity in XY (area of triangle test via rank)
    if np.linalg.matrix_rank(np.column_stack([X - X.mean(), Y - Y.mean()])) >= 2:
        tri = Triangulation(X, Y)

        # Surface through pareto points (semi-transparent)
        ax.plot_trisurf(
            tri, Z,
            alpha=0.25,
            linewidth=0.2,
            edgecolor="none",
            zorder=2
        )
    else:
        print("Pareto points are collinear in (exp_len, avg_pop); skipping surface.")
else:
    print("Not enough Pareto points to plot a surface; need >= 3.")

# Highlight Pareto-efficient mean points (same colors, bigger + thicker edge)
for exp, c in zip(labels, palette):
    sub = pareto3_df[pareto3_df["experiment"] == exp]
    if sub.empty:
        continue
    ax.scatter(
        sub["exp_len"], sub["avg_pop"], sub["avg_art_per_agent"],
        s=140,
        alpha=1.0,
        facecolors=mcolors.to_rgba(c, 1.0),
        edgecolors="black",
        linewidths=1.5,
        depthshade=False,
        zorder=10,
    )

# Ensure limits are set BEFORE droplines (important!)
ax.set_xlim(mean_df["exp_len"].min()-50, mean_df["exp_len"].max()+50)
ax.set_ylim(mean_df["avg_pop"].min()-1, mean_df["avg_pop"].max()+1)
ax.set_zlim(mean_df["avg_art_per_agent"].min()-1, mean_df["avg_art_per_agent"].max()+1)

# Add droplines for Pareto points
for _, row in pareto3_df.iterrows():
    add_droplines_3d(
        ax,
        row["exp_len"],
        row["avg_pop"],
        row["avg_art_per_agent"],
        alpha=1,
        lw=1.0
    )

ax.set_xlabel("Experiment Length (mean)")
ax.set_ylabel("Average Population Size (mean)")
ax.set_zlabel("Avg Artifacts per Agent (mean)")
ax.set_title("Maximize length, maximize pop & artifacts/agent")
ax.grid(alpha=0.3)

# Legend outside (optional)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True)
# ax.view_init(elev=20, azim=45)

plt.tight_layout()
os.makedirs(ROOT / "plots", exist_ok=True)
plt.savefig(ROOT / "plots" / "000_general_pareto_3d_means.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Time Series — Agents & Food

Two-panel time-series plot showing how population and food availability evolve over the course of each run.

- Shorter runs are padded with `NaN` so that per-condition means can be computed across runs of different lengths.
- When `SHOW_STD=True`: shaded ±1 std band around the mean.
- When `SHOW_STD=False`: individual run traces at low alpha, mean on top.

Saved as `000_agents_food.pdf`.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for base_name in data:
    agents_time_series = data[base_name]['agents_per_time']

    # Pad shorter arrays with NaN to align shapes
    max_len = max(arr.shape[0] for arr in agents_time_series)
    padded = np.full((len(agents_time_series), max_len), np.nan)
    for i, arr in enumerate(agents_time_series):
        padded[i, :arr.shape[0]] = arr

    mean_series = np.nanmean(padded, axis=0)
    std_series = np.nanstd(padded, axis=0)
    t = np.arange(max_len)

    ax1.plot(t, mean_series, label=base_name, color=color_map[base_name])
    if SHOW_STD:
        ax1.fill_between(t, mean_series - std_series, mean_series + std_series, alpha=0.3, color=color_map[base_name])
    else:
        for curve in agents_time_series:
            ax1.plot(np.arange(curve.shape[0]), curve, color=color_map[base_name], alpha=0.1)
    ax1.set_xlabel("Timestep")
    ax1.set_ylabel("Number of Agents")
    ax1.set_title(f"Agents per Timestep")
    ax1.legend()

    food_time_series = data[base_name]['food_per_time']

    # Pad shorter arrays with NaN to align shapes
    max_len = max(arr.shape[0] for arr in food_time_series)
    padded = np.full((len(food_time_series), max_len), np.nan)
    for i, arr in enumerate(food_time_series):
        padded[i, :arr.shape[0]] = arr

    mean_series = np.nanmean(padded, axis=0)
    std_series = np.nanstd(padded, axis=0)
    t = np.arange(max_len)

    ax2.plot(t, mean_series, label=PLOT_NAMES[base_name], color=color_map[base_name])
    if SHOW_STD:
        ax2.fill_between(t, mean_series - std_series, mean_series + std_series, alpha=0.3, color=color_map[base_name])
    else:
        for curve in food_time_series:
            ax2.plot(np.arange(curve.shape[0]), curve, color=color_map[base_name], alpha=0.1)
    ax2.set_xlabel("Timestep")
    ax2.set_ylabel("Total food")
    ax2.set_title(f"Food per Timestep")
    ax2.legend()
plt.tight_layout()
plt.savefig(ROOT / 'plots' / '000_agents_food.pdf',  dpi=300)
plt.show()

## Death Causes Over Time

Tracks the proportion of deaths attributable to **old age** vs **hunger** as a running cumulative fraction.

For each timestep `t`:
- `age_fraction[t] = cumsum(age_deaths[:t]) / cumsum(total_deaths[:t])`
- `hunger_fraction[t] = cumsum(hunger_deaths[:t]) / cumsum(total_deaths[:t])`

`NaN` is substituted with `0` before `0/0` cases (no deaths yet). Both panels share the x-axis and are bounded to `[0, 1]`.

Saved as `000_deaths.pdf`.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6))

for base_name, exps in experiments.items():
    age_deaths = [] 
    hunger_deaths = []
    total_deaths = []
    for exp in exps:
        worldlog_path = EXP_PATH / exp / 'open_gridworld.log'
        exp_len = get_last_ts(worldlog_path=worldlog_path)
        age = np.zeros(exp_len+1)
        hunger = np.zeros(exp_len+1)

        worldlog = load_worldlog(filepath=worldlog_path, agent_name=None, agent_tag=None)
        for line in worldlog:
            if line['event'] == "AGENT_DIED":
                t = int(line['timestamp'])
                if line['reason'] == 'hunger':
                    hunger[t] += 1
                elif line['reason'] == 'old age':
                    age[t] += 1

        age_deaths.append(age)
        hunger_deaths.append(hunger)
        total_deaths.append(age + hunger)


    max_len = max(arr.shape[0] for arr in age_deaths)
    age_padded = np.full((len(age_deaths), max_len), np.nan)
    hunger_padded = np.full((len(age_deaths), max_len), np.nan)
    for i, (age_arr, hunger_arr, total_arr) in enumerate(zip(age_deaths, hunger_deaths, total_deaths)):
        age_padded[i, :age_arr.shape[0]] = np.nan_to_num(np.cumsum(age_arr) / np.cumsum(total_arr), nan=0.0)
        hunger_padded[i, :hunger_arr.shape[0]] = np.nan_to_num(np.cumsum(hunger_arr) / np.cumsum(total_arr), nan=0.0)

    mean_age = np.nanmean(age_padded, axis=0)
    std_age = np.nanstd(age_padded, axis=0)
    mean_hunger = np.nanmean(hunger_padded, axis=0)
    std_hunger = np.nanstd(hunger_padded, axis=0)

    t = np.arange(max_len)

    ax[0].plot(t, mean_age, label=PLOT_NAMES[base_name], color=color_map[base_name])
    if SHOW_STD:
        ax[0].fill_between(t, mean_age - std_age, mean_age + std_age, alpha=0.3, color=color_map[base_name])
    else:
        for age_curve, total_curve in zip(age_deaths, total_deaths):
            curve = np.cumsum(age_curve) / np.cumsum(total_curve)
            curve = np.nan_to_num(curve, nan=0.0)
            ax[0].plot(np.arange(age_curve.shape[0]), curve, color=color_map[base_name], alpha=0.1)

    ax[1].plot(t, mean_hunger, label=PLOT_NAMES[base_name], color=color_map[base_name])
    if SHOW_STD:
        ax[1].fill_between(t, mean_hunger - std_hunger, mean_hunger + std_hunger, alpha=0.3, color=color_map[base_name])
    else:
        for hunger_curve, total_curve in zip(hunger_deaths, total_deaths):
            curve = np.cumsum(hunger_curve) / np.cumsum(total_curve)
            curve = np.nan_to_num(curve, nan=0.0)
            ax[1].plot(np.arange(hunger_curve.shape[0]), curve, color=color_map[base_name], alpha=0.1)


ax[0].set_xlabel("Timestep")
ax[0].set_ylabel("Normalized deaths by age")
ax[0].set_title(f"Deaths by Old Age per Timestep")
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].set_xlabel("Timestep")
ax[1].set_ylabel("Normalized deaths by hunger")
ax[1].set_title(f"Deaths by Hunger per Timestep")
# ax[1].legend()
ax[1].grid(alpha=0.3)

ax[0].set_ylim(-0.05, 1.05)
ax[1].set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig(ROOT / 'plots' / '000_deaths.pdf',  dpi=300)
plt.show()

## Action Rates Over Time

Tracks the cumulative per-agent rate of each action type over time.

**Actions tracked:** `give`, `take`, `reproduce`, `move`, `agents` (total agent-steps), `create_artifact`, `interact_artifact`, `give_artifact`, `pickup_artifact`, `drop_artifact`, `set_color`.

For each timestep:
- `action_data[t, a]` counts how many times action `a` was taken at timestep `t` (summed across all agents).
- `agents` column records total agent-steps, used for normalisation when `NORMALIZE=True`.
- Artifact interactions are cross-referenced from the world log (`ARTIFACT_INTERACTION` events).

When `NORMALIZE=True`, each cumulative curve is divided by cumulative agent-steps, giving a **per-agent rate**.

One subplot per action type. Saved as `000_actions.pdf`.

In [ ]:
ACTION_NAMES = ["give", "take", "reproduce", "move", "agents", "create_artifact", 
                "interact_artifact", "give_artifact", "pickup_artifact", "drop_artifact",
                "set_color"]
NORMALIZE = True
fig, ax = plt.subplots(len(ACTION_NAMES), 1, figsize=(10, 3 * len(ACTION_NAMES)))


for base_name, exps in experiments.items():
    all_data = []
    max_len = 0
    for exp in exps:
        worldlog_path = EXP_PATH / exp / 'open_gridworld.log'
        agent_files = glob.glob(str(EXP_PATH / exp / "agent_logs"/ "being*.jsonl"))
        last_ts = get_last_ts(worldlog_path=worldlog_path) + 1
        max_len = max(max_len, last_ts)

        action_data = np.zeros((last_ts+1, len(ACTION_NAMES)))

        for agent_file in agent_files:
            try:
                agent_data = load_agent_log(filepath=agent_file, reduce=False)
                for ts in agent_data:
                    time_idx = int(ts)
                    action = agent_data[ts]["action"]["action"]
                    params = agent_data[ts]["action"]["params"]
                    msg = agent_data[ts]['action']["message"]
                    if action == 'move':
                        if params['direction'] in ACTION_NAMES:
                            action_data[time_idx, ACTION_NAMES.index(params['direction'])] += 1

                    if action in ACTION_NAMES:
                        action_data[time_idx, ACTION_NAMES.index(action)] += 1

                    action_data[time_idx, ACTION_NAMES.index("agents")] += 1

                   
            except Exception as e:
                print(f"ERROR agent {agent_file}")
                # raise(e)
            
        worldlog = load_worldlog(filepath=worldlog_path, agent_name=None, agent_tag=None)
        for line in worldlog:
            if line['event'] == "ARTIFACT_INTERACTION":
                time_idx = int(line['timestamp'])
                action_data[time_idx, ACTION_NAMES.index("interact_artifact")] += 1

        all_data.append(action_data)

    for action_idx, action_name in enumerate(ACTION_NAMES):
        if NORMALIZE:
            column_data = [np.cumsum(exp_data[:, action_idx] / exp_data[:, ACTION_NAMES.index("agents")]) for exp_data in all_data]
        else:
            column_data = [np.cumsum(exp_data[:, action_idx]) for exp_data in all_data]

        padded = np.full((len(column_data), max_len+1), np.nan)
        for i, arr in enumerate(column_data):
            padded[i, :arr.shape[0]] = arr
        
        mean_action = np.nanmean(padded, axis=0)
        std_action = np.nanstd(padded, axis=0)
        max_len = mean_action.shape[0]

        ax[action_idx].plot(np.arange(max_len), mean_action, label=PLOT_NAMES[base_name], color=color_map[base_name])
        if SHOW_STD:
            ax[action_idx].fill_between(np.arange(max_len), mean_action - std_action, mean_action + std_action, alpha=0.3, color=color_map[base_name])
        else:
            for curve in column_data:
                ax[action_idx].plot(np.arange(curve.shape[0]), curve, color=color_map[base_name], alpha=0.1)

        ax[action_idx].set_title(f"Action: {action_name}")
        ax[action_idx].set_xlabel("Timestep")
        ax[action_idx].set_ylabel("Count")
        ax[action_idx].legend()
plt.tight_layout()
plt.savefig(ROOT / 'plots' / '000_actions.pdf',  dpi=300)
plt.show()